<a href="https://colab.research.google.com/github/Adamphoenix003/GNN-LinkPrediction/blob/main/GATandSAGE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -q

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


In [29]:
import torch
import torch.nn.functional as F
import numpy as np
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import GATConv
from torch_geometric.nn.models import GAE
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

In [30]:
class GATEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()

        self.conv1 = GATConv(
            in_channels,
            hidden_channels,
            heads=8,
            dropout=0.6
        )

        self.conv2 = GATConv(
            hidden_channels * 8,
            hidden_channels,
            heads=1,
            concat=False,
            dropout=0.6
        )

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        return x

In [41]:
def run_experiment(seed):

    torch.manual_seed(seed)
    np.random.seed(seed)

    transform = RandomLinkSplit(
        num_val=0.05,     # 5% validation
        num_test=0.10,    # 10% test
        is_undirected=True,
        add_negative_train_samples=True,
        neg_sampling_ratio=1.0  # equal pos/neg
    )

    train_data, val_data, test_data = transform(data)

    train_data = train_data.to(device)
    val_data = val_data.to(device)
    test_data = test_data.to(device)

    encoder = GATEncoder(dataset.num_features, 128)
    model = GAE(encoder).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

    # -------- TRAIN --------
    for epoch in range(300):
        model.train()
        optimizer.zero_grad()

        z = model.encode(train_data.x, train_data.edge_index)

        # Separate positive & negative edges
        pos_mask = train_data.edge_label == 1
        neg_mask = train_data.edge_label == 0

        pos_edge_index = train_data.edge_label_index[:, pos_mask].long()
        neg_edge_index = train_data.edge_label_index[:, neg_mask].long()

        loss = model.recon_loss(z, pos_edge_index, neg_edge_index)

        loss.backward()
        optimizer.step()

    # -------- EVALUATION --------
    model.eval()
    with torch.no_grad():
        z = model.encode(train_data.x, train_data.edge_index)

        def evaluate(split):
            out = model.decode(
                z,
                split.edge_label_index.long()
            ).view(-1).sigmoid()

            labels = split.edge_label

            auc = roc_auc_score(
                labels.cpu().numpy(),
                out.cpu().numpy()
            )

            ap = average_precision_score(
                labels.cpu().numpy(),
                out.cpu().numpy()
            )

            return auc, ap

        val_auc, val_ap = evaluate(val_data)
        test_auc, test_ap = evaluate(test_data)

    return val_auc, val_ap, test_auc, test_ap

In [43]:
val_aucs, val_aps = [], []
test_aucs, test_aps = [], []

for seed in range(1):
    val_auc, val_ap, test_auc, test_ap = run_experiment(seed)

    val_aucs.append(val_auc)
    val_aps.append(val_ap)
    test_aucs.append(test_auc)
    test_aps.append(test_ap)

    print(f"Run {seed}: Test AUC = {test_auc:.4f}")

print("\n===== Averaged Over 10 Runs =====")
print("Validation AUC:", np.mean(val_aucs))
print("Validation AP :", np.mean(val_aps))
print("Test AUC:", np.mean(test_aucs))
print("Test AP :", np.mean(test_aps))

Run 0: Test AUC = 0.8602

===== Averaged Over 10 Runs =====
Validation AUC: 0.8700429383105149
Validation AP : 0.8881143589217888
Test AUC: 0.86022345523874
Test AP : 0.8757057479988211


In [27]:
@torch.no_grad()
def test(data_split):
    model.eval()

    # Encode using ONLY train graph (no leakage)
    z = model.encode(train_data.x, train_data.edge_index)

    # Get predictions
    out = model.decode(z, data_split.edge_label_index).view(-1).sigmoid()
    labels = data_split.edge_label

    from sklearn.metrics import roc_auc_score, average_precision_score

    auc = roc_auc_score(labels.cpu().numpy(), out.cpu().numpy())
    ap = average_precision_score(labels.cpu().numpy(), out.cpu().numpy())

    return auc, ap

In [28]:
val_auc, val_ap = test(val_data)
test_auc, test_ap = test(test_data)

print("\nValidation AUC:", val_auc)
print("Validation AP :", val_ap)
print("Test AUC:", test_auc)
print("Test AP :", test_ap)


Validation AUC: 0.9405658604288049
Validation AP : 0.9531593295220856
Test AUC: 0.928102574812137
Test AP : 0.9336566373650602


In [58]:
class SAGEEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

        self.dropout = 0.3

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        return x
class MLPDecoder(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()

        self.lin1 = torch.nn.Linear(hidden_channels * 2, hidden_channels)
        self.lin2 = torch.nn.Linear(hidden_channels, 1)

    def forward(self, z, edge_index, sigmoid=True):
        src = z[edge_index[0]]
        dst = z[edge_index[1]]

        h = torch.cat([src, dst], dim=1)
        h = F.relu(self.lin1(h))
        h = self.lin2(h).view(-1)

        if sigmoid:
            return torch.sigmoid(h)
        else:
            return h

In [59]:
def sage_run(seed):

    torch.manual_seed(seed)
    np.random.seed(seed)

    transform = RandomLinkSplit(
        num_val=0.05,
        num_test=0.10,
        is_undirected=True,
        add_negative_train_samples=True,
        neg_sampling_ratio=1.0
    )

    train_data, val_data, test_data = transform(data)

    encoder = SAGEEncoder(dataset.num_features, 128)
    decoder = MLPDecoder(128)

    model = GAE(encoder, decoder).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    # -------- TRAIN --------
    for epoch in range(400):

        model.train()
        optimizer.zero_grad()

        z = model.encode(train_data.x, train_data.edge_index)

        pos_mask = train_data.edge_label == 1
        neg_mask = train_data.edge_label == 0

        pos_edge_index = train_data.edge_label_index[:, pos_mask]
        neg_edge_index = train_data.edge_label_index[:, neg_mask]

        loss = model.recon_loss(z, pos_edge_index, neg_edge_index)

        loss.backward()
        optimizer.step()

    # -------- EVALUATE --------
    model.eval()

    with torch.no_grad():

        z = model.encode(train_data.x, train_data.edge_index)

        def evaluate(split):
            logits = model.decoder(z, split.edge_label_index)
            probs = torch.sigmoid(logits)

            auc = roc_auc_score(
                split.edge_label.cpu().numpy(),
                probs.cpu().numpy()
            )

            ap = average_precision_score(
                split.edge_label.cpu().numpy(),
                probs.cpu().numpy()
            )

            return auc, ap

        val_auc, val_ap = evaluate(val_data)
        test_auc, test_ap = evaluate(test_data)

    return val_auc, val_ap, test_auc, test_ap

In [62]:
def run_sage_10():

    val_auc_list = []
    val_ap_list = []
    test_auc_list = []
    test_ap_list = []

    for seed in range(1):

        v_auc, v_ap, t_auc, t_ap = sage_run(seed)

        val_auc_list.append(v_auc)
        val_ap_list.append(v_ap)
        test_auc_list.append(t_auc)
        test_ap_list.append(t_ap)

        print(f"Run {seed+1}: "
              f"Val AUC={v_auc:.4f}, "
              f"Test AUC={t_auc:.4f}")

    print("\n===== FINAL AVERAGE (10 RUNS) =====")
    print("Validation AUC :", np.mean(val_auc_list))
    print("Validation AP  :", np.mean(val_ap_list))
    print("Test AUC       :", np.mean(test_auc_list))
    print("Test AP        :", np.mean(test_ap_list))


In [63]:
run_sage_10()

Run 1: Val AUC=0.7169, Test AUC=0.6952

===== FINAL AVERAGE (10 RUNS) =====
Validation AUC : 0.7168746114588905
Validation AP  : 0.714062839938408
Test AUC       : 0.6952334829996147
Test AP        : 0.6761547685074893
